# Git &amp; GitHub · Testing · CI/CD
## A Practical Workshop — Three Pillars of a Shared Codebase

**Amii — Engineering &amp; Performance**  ·  Amirreza Yasami

Hands-on sandbox: [https://github.com/amirrezayasami-amii/amii-workshop-sandbox](https://github.com/amirrezayasami-amii/amii-workshop-sandbox)

---

This notebook contains runnable cells that build small, self-contained scratch projects on disk (a Git repository for Pillar 1, a Python package for Pillar 2). They touch nothing else on your machine and can be deleted afterwards. Cells shown as fenced blocks inside **markdown** - anything that contacts a live remote
or GitHub - are provided for reference and are deliberately not executed during the session.


## Why we are here

Writing code is the easy part. Shipping it *as a team* - without losing work, breaking each other,
or shipping bugs - is the skill. This session builds the three habits that make that possible, on
one realistic project.

**The three pillars**

1. **Git &amp; GitHub** — collaborate without stepping on each other.
2. **Unit testing** — change code and *know* you did not break it.
3. **CI/CD (GitHub Actions)** — let automation enforce both on every push.

**The idea that ties them together.** A **pull request** (Pillar 1) runs your **tests** (Pillar 2)
automatically in **CI** (Pillar 3). Green means safe to merge; red blocks it. Everything here builds
toward that one workflow.

> **You do not need an ML background.** The sandbox is a small machine-learning service
> (data → model → API) used purely as something concrete to version, test, and ship. Every example
> is a plain function, a plain test, or a plain YAML file.


## How this session works

- **One project, three lenses.** A single small service carries all three pillars, so nothing is
  abstract.
- **Show, then do.** Each concept is followed by a command *you* run here, and a matching
  **planted exercise** already seeded in the sandbox for participants to try.
- **Read the graph first.** Whenever Git feels confusing, look at the branch graph *before*
  reaching for a command.


## Agenda

| 1 · Git &amp; GitHub | 2 · Unit testing | 3 · CI/CD |
| ------------------- | ---------------- | --------- |
| Mental model &amp; remotes | Why &amp; what to test | CI/CD concepts |
| Staging, `.gitignore`, commits | Anatomy &amp; assertions | Workflow anatomy |
| Branching model | Fixtures &amp; scope | Build matrix &amp; cache |
| Merge vs. rebase | Parametrization | Secrets &amp; permissions |
| Conflicts &amp; undo | -- | Coverage &amp; artifacts |
| Tags, LFS, secrets | Mocking &amp; the API | -- |
| Pull requests &amp; review | Coverage | -- |

**The sandbox project** — everything lives in one public repo: [https://github.com/amirrezayasami-amii/amii-workshop-sandbox](https://github.com/amirrezayasami-amii/amii-workshop-sandbox)

```
preprocessing.py     -> pandas/numpy transforms (the main unit-test target)
model.py / train.py  -> PyTorch MLP + artifact save/load
app.py               -> FastAPI /predict service
tests/               -> the pytest suite (fixtures, parametrization, mocks, API)
.github/workflows/   -> ci.yml (test matrix + coverage) + docker-publish.yml (build/publish)
config.py            -> hyperparameters (and deliberate conflict bait)
```

Data flow: **preprocess (pandas) → train (torch) → serve (FastAPI) → ship (Docker/GHCR)**.

Branches you will meet: `main` (stable, tag `v0.1.0`) · `staging` (default) ·
`feature/tune-lr` (conflict) · `feature/faster-epochs` (rebase) ·
`feature/clip-outliers` (failing test).


## Prerequisites and first-time setup

Git and GitHub are **not** the same thing. **Git** is the tool that versions your files and runs
entirely on your machine. **GitHub** is one of several websites (others: GitLab, Bitbucket) that
*host* Git repositories and add collaboration features. Git is fully usable with no GitHub account.

Verify your tooling (run this cell):

In [1]:
!git --version
import sys; print("Python:", sys.version.split()[0], "  (use 3.11 or 3.12 for the sandbox; torch has no 3.14 wheels yet)")

git version 2.54.0
Python: 3.12.13   (use 3.11 or 3.12 for the sandbox; torch has no 3.14 wheels yet)


### Configure your identity (one time, globally)

Every commit is permanently stamped with an author name and email. Use the **same email registered
on GitHub**, or your commits will not link to your profile.

```bash
git config --global user.name  "Your Name"
git config --global user.email "you@ualberta.ca"
```

A few defaults prevent whole classes of friction:

```bash
git config --global init.defaultBranch main    # name the first branch "main"
git config --global pull.ff only               # refuse surprise merge commits on pull
git config --global push.autoSetupRemote true  # first push auto-creates the upstream branch
git config --global core.editor "nano"         # editor for commit messages
```

Inspect configuration at any time:

```bash
git config --list --show-origin   # every setting and which file it came from
git config user.email             # a single value
```

> **Configuration has three levels**, each overriding the one above: **system** (`--system`, whole
> machine), **global** (`--global`, your account), **local** (`--local`, one repository). Use a
> local override when one project needs a different identity - e.g. a personal email on a work
> machine.

### Authentication is the #1 time-sink - settle it first

GitHub no longer accepts an account password on the command line. Use one of:

| Method | How it works | Notes |
| ------ | ------------ | ----- |
| **SSH** | Add your machine's public key to GitHub; clone with `git@github.com:…` URLs. | Best for a machine you control; no prompts after setup. |
| **HTTPS + token** | Clone with `https://` URLs; authenticate with a **Personal Access Token**. | Useful where SSH is firewalled. Store it with a credential helper. |

```bash
ssh-keygen -t ed25519 -C "you@ualberta.ca"   # generate a key pair
cat ~/.ssh/id_ed25519.pub                     # copy the PUBLIC key into GitHub -> Settings -> SSH keys
ssh -T git@github.com                         # test the connection
```

> **Only ever upload the public key** (the `.pub` file). Your private key must never be shared,
> copied to a shared machine, or committed.


# Pillar 1 — Git &amp; GitHub

## Why version control

Research and ML work is a sequence of experiments whose results must be *trustworthy* and
*reproducible*. Without disciplined version control this breaks down fast: code is emailed around;
directories named `final`, `final_v2`, `final_REALLY_final` accumulate; six months later nobody can
say which version produced which figure.

**Git** is a **distributed** version-control system: it records snapshots over time, lets many
people work in parallel, and lets you move backward and forward through history with confidence.
Because it is distributed, *every clone is a complete copy of the entire history* — you can commit,
branch, inspect, and diff completely offline.

For a research group, version control delivers four things otherwise hard to guarantee:

- **Provenance** — every line traces to *who* wrote it, *when*, and *why*.
- **Reproducibility** — regenerate any past result by checking out the exact commit that produced it.
- **Parallelism** — several people develop features and experiments simultaneously.
- **Safety** — committed *and pushed* work is extraordinarily hard to lose.

> **Warning.** That safety only applies to work you have **pushed**. A branch that lives only on
> your laptop is one disk failure — or one bad rebase — away from being gone. If you have made
> substantial local changes, push them (even to a work-in-progress branch) so a copy exists on the
> server.


## How Git works — a brief, accurate mental model

You do not need Git's internals to use it, but a small, accurate model prevents most confusion.
Most trouble comes not from the commands but from **not knowing what state the repository is in**.

### The three areas

Almost every Git command moves content between three places:

```
working tree  --( git add )-->  staging area  --( git commit )-->  repository (.git)
     ^                                                                    |
     +-----------------------( git restore )-----------------------------+
```

| Area | What it is |
| ---- | ---------- |
| **Working tree** | The actual files on disk that you edit. |
| **Staging area** (the *index*) | A holding zone where you assemble *exactly* what your next commit will contain. |
| **Repository** (`.git`) | The permanent, committed history of the project. |

The staging area is what distinguishes Git from simpler tools: it lets you craft a commit out of
*some* of your changes while leaving the rest for a later, separate commit — the basis of clean,
atomic history.

### The object model, commits, and branches

Under the hood Git is a content-addressed store of four object types: a **blob** (a file's
contents), a **tree** (a directory listing), a **commit** (one tree + metadata + parent link(s)),
and a **tag** (a named pointer to a commit). From this a few properties follow:

- A **commit** is a *complete snapshot*, not a diff - plus author, date, message, and a link to its
  **parent(s)**. Chaining parents forms the history, more precisely a **Directed Acyclic Graph
  (DAG)** once branches and merges exist.
- Every object is named by a **hash of its content**, so history is tamper-evident: change anything
  and every downstream hash changes. (This is also why *rewriting* history is possible - it produces
  new commits with new hashes.)
- A **branch** is nothing more than a lightweight, movable *pointer* to a commit. Creating one writes
  a single small file; that is why branching is cheap and encouraged.
- **HEAD** points to *where you are now* - normally the branch you have checked out. A "detached
  HEAD" means HEAD points directly at a commit rather than at a branch.

> **Hold on to one idea above all: branching is cheap and local.** A large share of Git's power comes
> from creating short-lived branches freely and integrating them back.


## A scratch repository for live demonstration

The cells below create a small throwaway repository so every command produces real output. Re-run
this cell at any time to start completely fresh.

In [2]:
import os, shutil
from pathlib import Path

REPO = Path.home() / "Desktop" / "git_demo" / "repo"
if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True)
os.chdir(REPO)                 # the kernel's working directory; every `!git ...` below runs here
print("Scratch repository:", REPO)

Scratch repository: /Users/amirreza.yasami/Desktop/git_demo/repo


In [3]:
# Initialise the repository and set a local identity so commits work without global config
!git init -q
!git branch -M main
!git config user.name  "Amirrreza Workshop Demo"
!git config user.email "amirreza.yasami@amii.ca"
!git status

On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


## The everyday solo loop: edit → stage → commit

`git status` is the single most useful command in Git — it reports what changed, what is staged, and
which branch you are on. Run it liberally.

In [4]:
from pathlib import Path
Path("README.md").write_text("# Demo Project\n")

!git add README.md
!git commit -q -m "docs: add README"
!git log --oneline

5eb9929 (HEAD -> main) docs: add README


### Staging with intent

A commit should be **one coherent idea**. The staging area lets you include *some* changes and leave
the rest. Below we create two files but stage only one.

> **Tip.** `git add -p` stages changes *hunk by hunk* (`y` / `n` / `s` to split / `q` to quit) — the
> way to split a messy working tree into several small, coherent commits. Small, single-purpose
> commits are easier to review, to `revert`, and to `cherry-pick`.

In [5]:
Path("train.py").write_text("lr = 0.001\nprint('training at', lr)\n")
Path("notes.txt").write_text("scratch notes\n")

!git add train.py
!git status -s     #  A = staged (train.py)   ?? = untracked (notes.txt)

A  train.py
?? notes.txt


In [6]:
!git commit -q -m "feat: add training script"
!git log --oneline

0f03272 (HEAD -> main) feat: add training script
5eb9929 docs: add README


### Inspecting history and differences

```bash
git log --oneline --graph --decorate --all   # compact, visual history of all branches
git show <commit>                            # exactly what one commit changed
git diff                                     # unstaged changes (working tree vs index)
git diff --staged                            # staged changes (index vs last commit)
git blame <file>                             # who last changed each line, and in which commit
```

In [7]:
# Modify a tracked file, then view the unstaged diff
Path("train.py").write_text("lr = 0.0005\nprint('training at', lr)\n")
!git diff

diff --git a/train.py b/train.py
index 34b1245..c25b369 100644
--- a/train.py
+++ b/train.py
@@ -1,2 +1,2 @@
-lr = 0.001
+lr = 0.0005
 print('training at', lr)


## Ignoring files with `.gitignore`

Git tracks *everything* you add — so tell it what to never track. Getting this right from the first
commit is critical for research repositories, which otherwise fill with large data files, model
checkpoints, and — most dangerously — secrets.

```
# Python
__pycache__/
*.pyc
.venv/

# Data & model artifacts (keep out of Git)
data/
*.csv
*.ckpt
*.pt

# Secrets & environment
.env
*.key

# Editor / OS noise
.vscode/
.DS_Store
```

> **Two gotchas.** `.gitignore` only affects files Git is **not already tracking** — if a file was
> already committed, ignoring it later does nothing until you `git rm --cached <file>` (keeps the
> file on disk, removes it from the index). And never rely on it for a secret you already pushed:
> once pushed, treat it as compromised and **rotate it**.

In [8]:
Path(".gitignore").write_text("notes.txt\n__pycache__/\n*.pyc\n.env\n")

# notes.txt is now ignored -> it disappears from status; only .gitignore is new
!git status -s
!git add .gitignore train.py
!git commit -q -m "chore: add .gitignore"
!git log --oneline

 M train.py
?? .gitignore
3a9af5e (HEAD -> main) chore: add .gitignore
0f03272 feat: add training script
5eb9929 docs: add README


## Commit messages — Conventional Commits

A consistent format makes history readable and machine-parseable.

```
<type>: <short summary in the imperative mood, <= 50 chars>

<optional body: WHY, not what — wrap at 72 columns>
```

Common types: `feat` (a new capability), `fix` (a bug fix), `test` (adding/changing tests),
`ci` (workflow changes), plus `docs`, `refactor`, `style`, `chore`. Example from the sandbox:
`test: add clip_outliers with a failing spec`.

> **Payoff.** A consistent history is skimmable, greppable, and lets tools auto-generate changelogs
> and version bumps. The whole seeded sandbox history follows this — run `git log --oneline`. The
> convention can be **enforced** in CI with `commitlint` via `pre-commit`.


## Branching and merging

A branch lets you develop a feature, experiment, or fix in isolation without disturbing a
known-good branch.

```bash
git branch                      # list local branches (current marked with *)
git switch -c feature/new-loss  # create AND switch to a new branch
git switch main                 # switch back
git branch -d feature/new-loss  # delete once merged
```

> **Note.** `git switch` (change branches) and `git restore` (discard file changes) are the modern,
> purpose-built commands. The older `git checkout` still does both jobs and appears throughout older
> tutorials. Adopt a **naming convention** — `feature/…`, `fix/…`, `experiment/…`, `docs/…` — so the
> branch list is self-documenting.

### Fast-forward merge
If the receiving branch has not moved, Git simply advances its pointer — no merge commit is created.

In [9]:
!git switch -c feature/greeting
Path("app.py").write_text("print('hello workshop')\n")
!git add app.py
!git commit -q -m "feat: add app greeting"

!git switch main
!git merge feature/greeting          # fast-forward: no new commit
!git log --oneline --graph --all

Switched to a new branch 'feature/greeting'
Switched to branch 'main'
Updating 3a9af5e..db6f192
Fast-forward
 app.py | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 app.py
* db6f192 (HEAD -> main, feature/greeting) feat: add app greeting
* 3a9af5e chore: add .gitignore
* 0f03272 feat: add training script
* 5eb9929 docs: add README


### `--no-ff` merge — preserve that a feature existed
When both branches have advanced, Git records a **merge commit** with two parents. `--no-ff`
*forces* a merge commit even when a fast-forward is possible, so the feature's commits stay grouped
under one merge and the branch's existence remains visible — far easier to read and to revert as a
unit. Many teams standardise on `--no-ff` into shared branches.

In [10]:
# A feature branch advances app.py ...
!git switch -c feature/subtitle
Path("app.py").write_text("print('hello workshop')\nprint('git & github')\n")
!git add app.py
!git commit -q -m "feat: add subtitle line"

# ... while main advances a different file (README) -> divergent histories
!git switch main
Path("README.md").write_text("# Demo Project\n\nA workshop sandbox.\n")
!git add README.md
!git commit -q -m "docs: expand README"

!git merge --no-ff feature/subtitle -m "Merge feature/subtitle into main"
!git log --oneline --graph --all

Switched to a new branch 'feature/subtitle'
Switched to branch 'main'
Merge made by the 'ort' strategy.
 app.py | 1 +
 1 file changed, 1 insertion(+)
*   eb56bb6 (HEAD -> main) Merge feature/subtitle into main
|\  
| * 393a75a (feature/subtitle) feat: add subtitle line
* | 8baf96f docs: expand README
|/  
* db6f192 (feature/greeting) feat: add app greeting
* 3a9af5e chore: add .gitignore
* 0f03272 feat: add training script
* 5eb9929 docs: add README


## A branching model for team &amp; production work

A shared model turns Git from a personal tool into a **release process**. The model below is a
lightly simplified [git-flow](https://nvie.com/posts/a-successful-git-branching-model/) - exactly
what the sandbox uses.

| Branch | Role |
| ------ | ---- |
| `main` | **Production.** Always the latest *stable* release; tagged releases live here (`v0.1.0`). Nothing lands until it is ready to ship. |
| `staging` | **The next release.** Finished features accumulate and stabilise here. |
| `feature/…` | **Work in progress.** Branch off `staging`, PR back into `staging` — never directly into `main`. |

The everyday flow:

```bash
git switch staging && git pull        # 1. sync the base branch
git switch -c feature/<your-name>     # 2. branch off staging
# ... work, commit in small focused steps ...
git push -u origin feature/<your-name> # 3. push
# 4. Open a PR into staging -> review -> merge
git switch staging && git pull        # 5. clean up
git branch -d feature/<your-name>
```

> **Branches can branch off branches.** Nothing forces every branch to start from `main`/`staging`.
> It is common to branch a small `fix/…` off a larger `feature/…`, merge it back, and roll the whole
> unit up into a release. Treating `main` as sacred, always-deployable production is the single habit
> that most improves stability — and it is how the wider industry works.


## Merge vs. rebase

Both integrate one branch's work into another, but they produce very different histories. This is
one of the biggest day-to-day "level-ups" in Git.

| Approach | What it does | When to use |
| -------- | ------------ | ----------- |
| **Merge** | Joins histories with a merge commit, preserving the true branching shape. | Integrating a finished branch into a **shared** branch; anything already public. |
| **Rebase** | Replays *your* commits on top of another branch → clean, linear history. | Updating/tidying **your own** feature branch before you open or update a PR. |

**The problem rebase solves.** If a feature keeps *merging* `staging` back in to stay current, the
history fills with back-and-forth merge commits:

```
*   Merge staging into feature (again)
|\
| * work on staging
* | your commit
|/
*   Merge staging into feature
|\
| * work on staging
* | your commit
|/
*   branch point
```

**Rebasing** instead lifts your commits and replays them on top of the latest `staging`:

```
Before rebase (feature branched off E):        After rebase (A B C replayed on top of G):

      A---B---C  feature                                         A'--B'--C'  feature
     /                                                          /
D---E---F---G  staging                             D---E---F---G  staging
```


In [11]:
# feature/rebase-me starts here and adds a commit
!git switch -c feature/rebase-me
Path("feature.txt").write_text("feature work A\n")
!git add feature.txt
!git commit -q -m "feat: A"

# main moves ahead in the meantime
!git switch main
Path("README.md").write_text("# Demo Project\n\nA workshop sandbox.\n\nMore docs.\n")
!git add README.md
!git commit -q -m "docs: more docs"

!git switch feature/rebase-me
print("--- BEFORE rebase: histories have diverged ---")
!git log --oneline --graph --all -6

Switched to a new branch 'feature/rebase-me'
Switched to branch 'main'
Switched to branch 'feature/rebase-me'
--- BEFORE rebase: histories have diverged ---
* e97101a (main) docs: more docs
| * 3a1220d (HEAD -> feature/rebase-me) feat: A
|/  
*   eb56bb6 Merge feature/subtitle into main
|\  
| * 393a75a (feature/subtitle) feat: add subtitle line
* | 8baf96f docs: expand README
|/  
* db6f192 (feature/greeting) feat: add app greeting


In [12]:
# Replay my commit on top of the latest main -> a single straight line
!git rebase main
print("--- AFTER rebase: my work sits linearly on top of main ---")
!git log --oneline --graph --all -6

Successfully rebased and updated refs/heads/feature/rebase-me.
--- AFTER rebase: my work sits linearly on top of main ---
* 75ce3bc (HEAD -> feature/rebase-me) feat: A
* e97101a (main) docs: more docs
*   eb56bb6 Merge feature/subtitle into main
|\  
| * 393a75a (feature/subtitle) feat: add subtitle line
* | 8baf96f docs: expand README
|/  
* db6f192 (feature/greeting) feat: add app greeting


> **Warning — rebasing rewrites history.** Because the replayed commits are *new* commits with new
> hashes, they are no longer the commits anyone else may have based work on. If a teammate has
> already pulled the originals, your force-push makes their next `git pull` **duplicate** everything
> and throw phantom conflicts. **Only rebase commits that are yours and unshared. Never rebase
> `main`/`staging`.**

**Pushing after a rebase.** Because you rewrote the branch, a normal `git push` is rejected. Force it
safely:

```bash
git push --force-with-lease    # preferred: refuses if someone else updated the branch
git push -f                    # blunt force: overwrites unconditionally (avoid)
```

**Escape hatch.** At any point during a rebase, `git rebase --abort` returns you to exactly where you
started. If you get lost, abort — nothing is lost.

**Interactive rebase — curate before you share.** `git rebase -i staging` opens an editor to reshape
your *own* commits:

```
pick   a1b2c3  feat: add clip_outliers
squash d4e5f6  fix typo            # fold into the commit above
reword 7a8b9c  test: add clip spec # edit the message
drop   0d0d0d  debug print         # remove entirely
```

The everyday combo: while working, `git commit --fixup <sha>` marks a fix; before the PR,
`git rebase -i --autosquash staging` folds each fixup into its target automatically — a clean,
reviewable series instead of "wip, wip, oops".

> **The live exercise:** in the sandbox, `feature/faster-epochs` is one commit behind `staging` —
> exactly this shape. Rebase it, then `git push --force-with-lease`.


## Working with GitHub

Everything so far has been entirely local. GitHub introduces a **remote**: a shared copy of the
repository you push to and pull from. By convention the primary remote is called `origin`.

```bash
git remote -v                                       # list configured remotes
git remote add origin git@github.com:you/repo.git   # link a local repo to GitHub
git push -u origin main                             # first push; -u sets the upstream
```

| Command | What it does |
| ------- | ------------ |
| `git push` | Upload your local commits to the remote. |
| `git fetch` | Download remote changes **without** touching your working files — for inspection first. |
| `git pull` | `git fetch` **plus** integrate the changes into your current branch, in one step. |

> **fetch vs. pull — the distinction that avoids surprises.** `fetch` is safe and read-only: it
> updates `origin/main` but leaves your working tree alone, so you can inspect `git log origin/main`
> before integrating. `pull` = `fetch` + integrate immediately. When unsure, **fetch first, look,
> then merge/rebase.**

### Visualising branch history
Reading the commit graph is a skill; good tools make it effortless.

- **GitHub network graph** — every repo has an interactive branch/commit graph at `…/network`.
- **GitLens** (VS Code / Cursor) — hover any line to see the last commit that changed it, with
  click-through to GitHub. Superb for "who changed this and why".
- **GitHub Desktop** / **Insights → Network** — the same history, graphically.

A handy terminal alias:

```bash
git config --global alias.graph "log --oneline --graph --decorate --all"
git graph
```


## The pull-request workflow

This is the core of team collaboration, and what the sandbox uses.

1. **Sync** your base branch: `git switch staging && git pull`.
2. **Branch** for your task: `git switch -c feature/add-fsdp-example`.
3. **Work**, making focused commits.
4. **Push**: `git push -u origin feature/add-fsdp-example`.
5. **Open a Pull Request** into the base branch. Describe *what* changed and *why*; link any issue.
   Open it as a **draft** if not ready.
6. **Review**: teammates comment, request changes, or approve. Push more commits to the same branch
   to address feedback — the PR updates automatically.
7. **Merge** once approved and checks pass (the merge-commit option corresponds to `--no-ff`).
8. **Clean up**: delete the merged branch; `git switch staging && git pull`.

> **Local merge vs. Pull Request.** A local `git merge` is right for folding *your own* branches
> together. A **PR** is right for landing work in a **shared** branch: it exists to get the change
> *reviewed* and leaves a durable, searchable record — every closed PR is a self-contained "here is a
> chunk of work that shipped, and why." A good habit even solo: land on `main`/`staging` through a
> PR. Most teams also enable **branch protection**, which *requires* a PR anyway.

### Fork vs. branch
| Model | When to use |
| ----- | ----------- |
| **Shared repository** (branches) | You have write access. Everyone pushes branches to the *same* repo and opens PRs internally — the typical internal-team setup. |
| **Fork &amp; pull** | You are an outside contributor without write access. You **fork** (a server-side copy), push to your fork, and open a PR back to the original ("upstream"). The standard way to contribute to open source. |

Keep a fork current by adding an `upstream` remote:

```bash
git remote add upstream git@github.com:Amii-Engineering/amii-docs.git
git fetch upstream
git switch main
git merge upstream/main
```

### Reviewing code well — including AI-generated code
- **Keep PRs small and focused.** A reviewer reasons carefully about 200 lines; a 1,000-line PR gets
  a rubber-stamp.
- **Review the *why*, not just the *what*.** The description should let a reviewer judge whether the
  *approach* is sound.
- **Be specific and kind.** Comment on the code, not the author; prefer questions and suggestions.

A growing share of code now originates from AI assistants and agents. The centre of gravity of review
has shifted to two questions: **(1) Did it build the right thing?** — verify the output matches what
was *desired*, and that a *test* proves it. **(2) Does the change fit the whole?** — confirm it
belongs where it landed and is consistent with the surrounding architecture. Git is the safety layer:
keep agent work on a branch, commit in small steps, and review the diff before pushing.


## Resolving merge conflicts

A **conflict** arises when two branches change the *same lines* of a file differently. Git cannot
know which is correct, so it pauses and asks you. Conflicts are normal collaboration, not an error.

We reproduce the sandbox's planted conflict: two branches editing the same `LEARNING_RATE` line in
`config.py`.

In [13]:
# Baseline config on main
Path("config.py").write_text("LEARNING_RATE = 0.001\nEPOCHS = 10\n")
!git add config.py
!git commit -q -m "feat: add config"

# feature/tune-lr lowers the learning rate
!git switch -c feature/tune-lr
Path("config.py").write_text("LEARNING_RATE = 0.0005\nEPOCHS = 10\n")
!git add config.py
!git commit -q -m "perf: lower learning rate for stability"

# meanwhile main RAISES it on the very same line
!git switch main
Path("config.py").write_text("LEARNING_RATE = 0.01\nEPOCHS = 10\n")
!git add config.py
!git commit -q -m "perf: raise learning rate to train faster"
print("--- two branches now disagree on the LEARNING_RATE line ---")

Switched to a new branch 'feature/tune-lr'
Switched to branch 'main'
--- two branches now disagree on the LEARNING_RATE line ---


In [14]:
# Attempt the merge -> CONFLICT (a non-zero exit code is expected)
!git merge feature/tune-lr; echo "merge exit code: $?"
!git status -s

Auto-merging config.py
CONFLICT (add/add): Merge conflict in config.py
Automatic merge failed; fix conflicts and then commit the result.
merge exit code: 1
AA config.py
A  feature.txt


In [15]:
# Git has written conflict markers into the file:
!cat config.py

<<<<<<< HEAD
LEARNING_RATE = 0.01
LEARNING_RATE = 0.0005
>>>>>>> feature/tune-lr
EPOCHS = 10


Between `<<<<<<< HEAD` and `=======` is *your* (current-branch) version; between `=======` and
`>>>>>>>` is the *incoming* version. Resolve by editing the region into the final result and deleting
all three marker lines, then stage and commit.

> **Editor buttons** like VS Code's "Accept Current / Incoming / Both" are convenient for trivial
> conflicts, but for real projects prefer resolving **by hand** — the correct result is frequently a
> careful *blend* of both changes, and only reading the code closely will get it right. Here we
> settle on `0.001`.

In [16]:
# Resolve by hand, then complete the merge
Path("config.py").write_text("LEARNING_RATE = 0.001\nEPOCHS = 10\n")
!git add config.py
!git commit -q --no-edit          # completes the merge with the pre-filled message
!git log --oneline --graph -4
print("\nResolved file:")
!cat config.py

*   9155ebf (HEAD -> main) Merge branch 'feature/tune-lr'
|\  
| * 0399db0 (feature/tune-lr) perf: lower learning rate for stability
| * b2e38a3 (feature/rebase-me) feat: add config
| * 75ce3bc feat: A

Resolved file:
LEARNING_RATE = 0.001
EPOCHS = 10


> `git merge --abort` bails out cleanly at any point. Enable
> `git config --global rerere.enabled true` so Git *remembers* a resolution and replays it if the
> same conflict recurs during a long rebase.


## Undoing things

Almost nothing that has been *committed* is truly lost, but the correct command depends entirely on
the situation. Choosing the wrong one — especially `reset --hard` — is the most common way beginners
lose work.

| You want to… | Command |
| ------------ | ------- |
| Discard unstaged changes to a file | `git restore <file>` |
| Unstage a file (keep the edits) | `git restore --staged <file>` |
| Amend the **last** commit (message or content) | `git commit --amend` |
| Undo the last commit, **keep** changes staged | `git reset --soft HEAD~1` |
| Undo the last commit, unstage (keep files) | `git reset HEAD~1` *(mixed — the default)* |
| Undo the last commit, **discard** changes | `git reset --hard HEAD~1`  ⚠ |
| Undo a commit **already pushed / shared** | `git revert <commit>` |

> `HEAD~1` means "one commit before HEAD"; `HEAD~3` rewinds three. **`git reset` rewrites history** —
> safe only on unshared commits. **`git revert` creates a *new* commit** that cancels an old one,
> leaving history intact — the correct choice for anything already pushed. `git reset --hard`
> permanently discards *uncommitted* work; when in doubt, `git stash` first.

In [17]:
# restore: discard an unstaged edit
Path("README.md").write_text("OOPS I broke the readme\n")
!git restore README.md
print("Restored README:")
!cat README.md

Restored README:
# Demo Project

A workshop sandbox.

More docs.


In [18]:
# reset --soft: undo the last commit but keep the change staged
Path("scratch.txt").write_text("temporary\n")
!git add scratch.txt
!git commit -q -m "temp: scratch commit"
!git reset --soft HEAD~1
print("After soft reset, scratch.txt is staged again (A):")
!git status -s

After soft reset, scratch.txt is staged again (A):
A  scratch.txt


In [19]:
# revert: safe undo of a (pretend-pushed) commit -> a NEW inverse commit
!git commit -q -m "temp: scratch commit"       # re-commit the staged file
!git revert --no-edit HEAD
!git log --oneline -3

[main 9c3f998] Revert "temp: scratch commit"
 Date: Tue Aug 4 13:28:40 2026 -0600
 1 file changed, 1 deletion(-)
 delete mode 100644 scratch.txt
9c3f998 (HEAD -> main) Revert "temp: scratch commit"
268c820 temp: scratch commit
9155ebf Merge branch 'feature/tune-lr'


### The reflog — your safety net
Git records every movement of `HEAD` in the **reflog**. Even after a mistaken `reset` or a deleted
branch, the commit almost always still exists (typically for ~30 days) and can be recovered. Before
assuming work is gone, run `git reflog`.

In [20]:
# "Lose" a commit with a hard reset ...
!git reset --hard HEAD~1
print("--- reflog: every HEAD position, newest first ---")
!git reflog -5

HEAD is now at 268c820 temp: scratch commit
--- reflog: every HEAD position, newest first ---
268c820 (HEAD -> main) HEAD@{0}: reset: moving to HEAD~1
9c3f998 HEAD@{1}: revert: Revert "temp: scratch commit"
268c820 (HEAD -> main) HEAD@{2}: commit: temp: scratch commit
9155ebf HEAD@{3}: reset: moving to HEAD~1
5c2c1fb HEAD@{4}: commit: temp: scratch commit


In [21]:
# ... then recover it onto a new branch using its reflog position.
# NOTE: git reflog's HEAD@{N} braces collide with Jupyter '!' brace-interpolation,
# so we resolve the reference to a commit hash in plain Python first.
import subprocess
lost = subprocess.check_output(["git", "rev-parse", "HEAD@{1}"], text=True).strip()
!git switch -c recovered {lost}
!git log --oneline -2
!git switch main

Switched to a new branch 'recovered'
9c3f998 (HEAD -> recovered) Revert "temp: scratch commit"
268c820 (main) temp: scratch commit
Switched to branch 'main'


### Set work aside &amp; grab single commits

**Stash** parks uncommitted changes so you can switch context with a clean tree, then restores them.
**Cherry-pick** copies a single commit onto the current branch — the fix for "committed to the wrong
branch". **Bisect** binary-searches history to find the commit that introduced a bug.

```bash
git stash -u            # park changes, including untracked files
git stash list
git stash pop           # reapply the most recent stash and drop it

git switch feature/right
git cherry-pick <sha>   # copy one commit onto this branch

git bisect start; git bisect bad; git bisect good <old-sha>   # guided binary search
```

In [22]:
# stash demo: park a change, get a clean tree, then restore it
Path("README.md").write_text("WIP edit, not ready to commit\n")
!git stash
print("After stash — working tree is clean:")
!git status -s
!git stash pop
print("\nAfter pop — the WIP edit is back:")
!git status -s
!git restore README.md          # tidy up

Saved working directory and index state WIP on main: 268c820 temp: scratch commit
After stash — working tree is clean:
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   README.md

no changes added to commit (use "git add" and/or "git commit -a")
Dropped refs/stash@{0} (4e610bde8f23b994ebafc611768c28ea228c6fa3)

After pop — the WIP edit is back:
 M README.md


## Common problems and how to fix them

**Working alone**

- **"Detached HEAD".** You ran `git checkout <hash>` and are viewing an old commit, not a branch;
  new commits here can be lost. To keep work: `git switch -c my-branch`. To leave: `git switch main`.
- **Committed to the wrong branch.** If you meant to start a feature: `git switch -c feature/intended`,
  then `git switch main && git reset --hard origin/main` (only if `main` is not yet pushed). If the
  feature branch exists: `git switch feature/existing`, `git cherry-pick <stray-sha>`, then roll
  `main` back.
- **Committed a large file or a secret.** Removing it in a *new* commit is not enough — it stays in
  history. For the last commit: `git rm --cached bigfile && git commit --amend`. For deeper history:
  `git filter-repo` or BFG. If a **secret** leaked, **rotate it immediately**.
- **Diverged from `origin/main`.** Integrate with `git pull` (merge) or `git pull --rebase` (linear),
  then push.
- **Orphan / disconnected branches.** `git checkout --orphan` creates a second, disconnected DAG that
  can never cleanly merge back. Avoid it — branch off the existing history instead.

**Working in a team**

- **Push rejected (`! [rejected] … fetch first`).** Someone pushed before you. `git pull` (resolve
  conflicts), then `git push`. **Do not** force-push a shared branch to sidestep this.
- **The temptation to force-push.** `git push --force` can erase teammates' commits. If you truly must
  (after rebasing *your own* branch), use `git push --force-with-lease`. Never force-push `main` /
  `staging` — branch protection usually forbids it outright.
- **A merged PR introduced a bug.** Use `git revert` (a new inverse commit) rather than rewriting
  shared history. The GitHub UI's **"Revert"** button on a merged PR opens a ready-made revert PR.


## Large files and datasets

For ML work this deserves special attention. **Git stores the full history of every file forever.**
Commit a 2 GB checkpoint, change it ten times, and *every* clone now carries ~20 GB that cannot be
removed without rewriting history. Binaries also cannot be diffed or merged. **Keep data and
checkpoints out of Git** and store the payload on an object store, a shared filesystem, or the
cluster's project space.

When large files genuinely must be versioned, use **[Git LFS](https://git-lfs.com)** (Large File
Storage): a tiny text *pointer* lives in Git while the real bytes live on a separate LFS server, so
clones stay small.

```bash
git lfs install                    # one-time setup per machine
git lfs track "*.pt" "*.ckpt"      # choose which patterns LFS handles
git add .gitattributes             # LFS records tracked patterns here — commit it
git add model.pt
git commit -m "Add trained checkpoint via LFS"
git push                           # the pointer goes to Git; the bytes go to LFS storage
```

For heavier, pipeline-style data versioning (datasets tied to experiments), consider a dedicated tool
such as **[DVC](https://dvc.org)**.

> **Never commit secrets.** API keys and `.env` files do not belong in Git. If one is pushed, it is
> compromised the moment it lands - **rotate it immediately.**


## Tags &amp; releases — pin the exact code behind a result

A **tag** marks a permanent, named point in history - ideal for "the exact code behind this result."
Annotated tags carry a message and author.

```bash
git tag -a v1.0 -m "NeurIPS 2026 submission"
git push origin v1.0        # tags are not pushed by default; push them explicitly
```

Once pushed, the tag appears under **Releases/Tags**; anyone can download a `.zip`/`.tar.gz` of the
repository *exactly as it was at that commit*. Link the release (or its commit hash) in your paper and
future readers can retrieve precisely the code behind your numbers.

> **Version the environment, not just the code.** Commit your `pyproject.toml` / lockfile /
> `requirements.txt` so a tagged checkout rebuilds the same environment. The sandbox's `main` carries
> `v0.1.0`; the CD pipeline also publishes a Docker image on `v*` tags.


## Best practices &amp; high-leverage habits

**For researchers**
- **Commit early, commit often, in logical units.** Atomic commits are easier to review, revert, and
  understand later.
- **Write meaningful messages** — state *what* changed and *why*; follow Conventional Commits.
- **Never commit secrets or data.** Use `.gitignore` and environment variables; rotate anything that
  leaks.
- **Branch per task**; keep production branches always working.
- **Integrate frequently.** Pull before you push; rebase/merge from the base branch often.
- **Push work you care about** — a local-only branch is not backed up.

**Tips &amp; tricks**
- Fix an earlier commit cleanly with `git commit --fixup <sha>` + `git rebase -i --autosquash`.
- Grab one commit with `git cherry-pick <sha>`; find a regression with `git bisect`.
- Always prefer `git push --force-with-lease` over `git push --force`.
- Enable `git config --global rerere.enabled true` so you do not re-solve the same conflict twice.

> **Bridge to Pillars 2 &amp; 3.** Protect `main`/`staging` with **branch protection**: no direct
> pushes, no force-push, and **required status checks** - a PR cannot merge until CI is green. So a
> trustworthy PR needs *tests* run by a *robot*. That is what comes next.


# Pillar 2 — Unit testing with pytest

## Why we test — and what to test

- **Confidence to change** — refactor, or accept an agent's diff, and know in *seconds* whether you
  broke something.
- **Executable documentation** — a test states intended behaviour precisely.
- **Fast feedback** — a bug caught in a unit test costs seconds, not hours.

pytest discovers any file named `test_*.py`, and inside it any function named `test_*`. A test
"passes" if it runs without raising `AssertionError`.


## A scratch project for live demonstration

The cells below build a small package that mirrors the sandbox's `preprocessing.py` and a subset of
its test suite, so every command runs and produces real output.

> **Kernel note.** These cells use `pandas`/`numpy` to match the sandbox exactly. Run this notebook
> on a **Python 3.11 or 3.12** kernel where those install cleanly (torch has no 3.14 wheels yet).

In [23]:
import os, shutil, sys, subprocess
from pathlib import Path

LAB = Path.home() / "Desktop" / "pytest_demo"
if LAB.exists():
    shutil.rmtree(LAB)
(LAB / "tests").mkdir(parents=True)
os.chdir(LAB)

# Install the tools this section uses (safe to re-run; harmless if already present)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "pandas", "pytest", "pytest-cov"], check=False)
print("Test lab:", LAB)

Test lab: /Users/amirreza.yasami/Desktop/pytest_demo



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


### The code under test — the sandbox's real `preprocessing.py`

In [24]:
%%writefile preprocessing.py
"""Data preprocessing utilities."""

import os

import numpy as np
import pandas as pd


def drop_missing(df: pd.DataFrame, columns=None) -> pd.DataFrame:
    """Drop rows containing missing values (optionally within a subset of columns)."""
    return df.dropna(subset=columns).reset_index(drop=True)


def normalize(df: pd.DataFrame, columns) -> pd.DataFrame:
    """Min-max normalize the given columns into [0, 1]. Constant columns map to zeros."""
    out = df.copy()
    for col in columns:
        col_min, col_max = out[col].min(), out[col].max()
        span = col_max - col_min
        out[col] = 0.0 if span == 0 else (out[col] - col_min) / span
    return out


def standardize(df: pd.DataFrame, columns) -> pd.DataFrame:
    """Standardize columns to zero mean and unit variance. Constant columns map to zeros."""
    out = df.copy()
    for col in columns:
        mean, std = out[col].mean(), out[col].std(ddof=0)
        out[col] = 0.0 if std == 0 else (out[col] - mean) / std
    return out


def encode_labels(series: pd.Series):
    """Encode a categorical series into integer labels; return (encoded, sorted classes)."""
    classes = np.sort(series.dropna().unique())
    lookup = {value: idx for idx, value in enumerate(classes)}
    return series.map(lookup).to_numpy(), classes


def load_dataset(path: str) -> pd.DataFrame:
    """Load a CSV dataset from disk (raises FileNotFoundError if missing)."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"No such dataset: {path}")
    return pd.read_csv(path)


def save_dataset(df: pd.DataFrame, path: str) -> None:
    """Write a DataFrame to CSV without the index."""
    df.to_csv(path, index=False)


def preprocess_pipeline(path: str, normalize_columns) -> pd.DataFrame:
    """Load a dataset, drop missing rows, and normalize the given columns."""
    df = load_dataset(path)
    df = drop_missing(df)
    return normalize(df, normalize_columns)


def train_test_split(df: pd.DataFrame, test_size: float = 0.2, seed: int = 0):
    """Split a DataFrame into (train, test) with reset indices."""
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1 (exclusive)")
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(df))
    n_test = int(round(len(df) * test_size))
    test = df.iloc[indices[:n_test]].reset_index(drop=True)
    train = df.iloc[indices[n_test:]].reset_index(drop=True)
    return train, test

Writing preprocessing.py


### Fix the #1 pytest gotcha up front — the import path

With tests in `tests/` and the module at the project root, **bare `pytest`** adds `tests/` to
`sys.path`, **not** the root - so `import preprocessing` fails with `ModuleNotFoundError`. It often
*seems* to work under `python -m pytest` (which adds the cwd), masking the bug until Docker/CI runs
bare `pytest`. The fix is a one-time config block:

In [25]:
%%writefile pyproject.toml
[tool.pytest.ini_options]
pythonpath = ["."]        # put the project ROOT on sys.path
testpaths  = ["tests"]    # where tests live

Writing pyproject.toml


## Anatomy of a test, fixtures, and parametrization

A pytest test is a function named `test_*` with a plain `assert`, following **Arrange → Act →
Assert**. A **fixture** is a function whose return value is *injected* into any test that names it as
a parameter - each test gets a **fresh** copy, so tests never leak state. Shared fixtures go in
`tests/conftest.py`, which pytest discovers automatically (you never import it).

In [26]:
%%writefile tests/conftest.py
"""Shared pytest fixtures — discovered automatically across the suite."""

import numpy as np
import pandas as pd
import pytest


@pytest.fixture
def sample_df():
    """A small, clean DataFrame used across many tests."""
    return pd.DataFrame(
        {
            "a": [1.0, 2.0, 3.0, 4.0],
            "b": [10.0, 20.0, 30.0, 40.0],
            "label": ["x", "y", "x", "z"],
        }
    )


@pytest.fixture
def df_with_nans():
    """A DataFrame containing missing values."""
    return pd.DataFrame({"a": [1.0, np.nan, 3.0], "b": [4.0, 5.0, np.nan]})


@pytest.fixture
def csv_file(tmp_path):
    """A real CSV on disk in a temp dir (uses the built-in tmp_path fixture); yields its path."""
    path = tmp_path / "data.csv"
    path.write_text("a,b,label\n1.0,10.0,x\n2.0,20.0,y\n3.0,30.0,x\n4.0,40.0,z\n")
    return str(path)

Writing tests/conftest.py


In [27]:
%%writefile tests/test_preprocessing.py
"""Fixtures, assertions, and parametrization on the pure transforms."""

import pandas as pd
import pytest

from preprocessing import drop_missing, normalize, train_test_split

# --- a plain test using the sample_df fixture (Arrange -> Act -> Assert) ---
def test_normalize_scales_to_unit_range(sample_df):
    result = normalize(sample_df, ["a"])          # Act
    assert result["a"].min() == 0.0               # Assert
    assert result["a"].max() == 1.0


def test_normalize_does_not_mutate_input(sample_df):
    original = sample_df["a"].tolist()
    normalize(sample_df, ["a"])
    assert sample_df["a"].tolist() == original     # input untouched


# --- assertions on floats: never bare ==, use pytest.approx ---
def test_normalize_exact_values(sample_df):
    result = normalize(sample_df, ["a"])
    assert result["a"].tolist() == pytest.approx([0.0, 1 / 3, 2 / 3, 1.0])


# --- parametrization: one body, many cases; each reports separately ---
@pytest.mark.parametrize(
    "subset, expected_len",
    [
        (None, 1),        # any NaN drops the row
        (["a"], 2),       # only column "a" considered
        (["a", "b"], 1),  # both considered
    ],
)
def test_drop_missing_subset(df_with_nans, subset, expected_len):
    assert len(drop_missing(df_with_nans, columns=subset)) == expected_len


# --- asserting an expected error, and checking its message with match= ---
@pytest.mark.parametrize("bad_size", [0, 1, -0.1, 1.5])
def test_train_test_split_rejects_invalid_size(sample_df, bad_size):
    with pytest.raises(ValueError, match="between 0 and 1"):
        train_test_split(sample_df, test_size=bad_size)


def test_train_test_split_is_reproducible(sample_df):
    t1, _ = train_test_split(sample_df, test_size=0.5, seed=7)
    t2, _ = train_test_split(sample_df, test_size=0.5, seed=7)
    assert t1["a"].tolist() == t2["a"].tolist()

Writing tests/test_preprocessing.py


**Key ideas above.** A test simply *asks* for `sample_df` and pytest runs the fixture and passes
the result in. `@pytest.mark.parametrize` runs the body once per row and reports each case
separately, so a failure tells you *exactly* which input broke. Use `pytest.approx` for floats, and
`pytest.raises(..., match=...)` to assert an error *and* verify its message so a different error can
not pass silently.

## Mocking — isolate code from the outside world

Some code touches things that are slow, non-deterministic, or unavailable in a test: the filesystem,
a network API, a database. **Mocking** replaces those with fakes you control, so the test exercises
*your* logic - not pandas' CSV parser or the disk.

In [28]:
%%writefile tests/test_io_mocks.py
"""Mocking the I/O boundary so no real file is ever read."""

import pandas as pd
import pytest
from unittest.mock import MagicMock, patch

import preprocessing
from preprocessing import load_dataset, preprocess_pipeline, save_dataset


# --- @patch decorator: replace read_csv and exists (decorators apply bottom-up) ---
@patch("preprocessing.pd.read_csv")
@patch("preprocessing.os.path.exists", return_value=True)
def test_load_dataset_reads_existing_file(mock_exists, mock_read_csv, sample_df):
    mock_read_csv.return_value = sample_df
    result = load_dataset("whatever.csv")
    mock_exists.assert_called_once_with("whatever.csv")     # assert HOW it was called
    mock_read_csv.assert_called_once_with("whatever.csv")
    pd.testing.assert_frame_equal(result, sample_df)


# --- the error path needs no real file at all ---
@patch("preprocessing.os.path.exists", return_value=False)
def test_load_dataset_missing_file_raises(mock_exists):
    with pytest.raises(FileNotFoundError, match="No such dataset"):
        load_dataset("missing.csv")


# --- monkeypatch style (pytest-native; auto-reverts, no decorator stacking) ---
def test_load_dataset_with_monkeypatch(monkeypatch, sample_df):
    fake_read = MagicMock(return_value=sample_df)
    monkeypatch.setattr(preprocessing.os.path, "exists", lambda p: True)
    monkeypatch.setattr(preprocessing.pd, "read_csv", fake_read)
    result = load_dataset("data.csv")
    fake_read.assert_called_once_with("data.csv")
    pd.testing.assert_frame_equal(result, sample_df)


# --- verify a call without a real object (spec catches typos) ---
def test_save_dataset_calls_to_csv_without_index():
    mock_df = MagicMock(spec=pd.DataFrame)
    save_dataset(mock_df, "out.csv")
    mock_df.to_csv.assert_called_once_with("out.csv", index=False)


# --- mock load_dataset, exercise the REAL transforms downstream ---
@patch("preprocessing.load_dataset")
def test_preprocess_pipeline_uses_loaded_data(mock_load):
    mock_load.return_value = pd.DataFrame({"a": [0.0, 5.0, 10.0], "b": [1.0, 2.0, 3.0]})
    result = preprocess_pipeline("any.csv", normalize_columns=["a"])
    mock_load.assert_called_once_with("any.csv")
    assert result["a"].tolist() == pytest.approx([0.0, 0.5, 1.0])   # "a" normalized; "b" untouched
    assert result["b"].tolist() == [1.0, 2.0, 3.0]

Writing tests/test_io_mocks.py


### Testing the API — `TestClient` hits real endpoints in-process
The sandbox's `tests/test_api.py` runs the FastAPI app with no server and no ports. A tiny
random-weight model is saved to `tmp_path`, so no real training is needed:

```python
from fastapi.testclient import TestClient

def test_predict_returns_a_number(monkeypatch, trained_model_dir):
    client, _ = make_client(monkeypatch, trained_model_dir)
    resp = client.post("/predict", json=EXAMPLE)
    assert resp.status_code == 200
    assert isinstance(resp.json()["prediction"], float)

def test_predict_rejects_missing_field(monkeypatch, trained_model_dir):
    resp = client.post("/predict", json=bad)   # a field is missing
    assert resp.status_code == 422             # pydantic validation error
```


## Run the suite

The suite's pass/fail is pytest's **exit code** — the same signal CI uses in Pillar 3.

In [29]:
!{sys.executable} -m pytest -v

============================= test session starts ==============================
platform darwin -- Python 3.12.13, pytest-9.1.1, pluggy-1.6.0 -- /Users/amirreza.yasami/.venvs/amii-workshop/bin/python
cachedir: .pytest_cache
rootdir: /Users/amirreza.yasami/Desktop/pytest_demo
configfile: pyproject.toml
testpaths: tests
plugins: cov-7.1.0
collected 16 items                                                             

tests/test_io_mocks.py::test_load_dataset_reads_existing_file PASSED     [  6%]
tests/test_io_mocks.py::test_load_dataset_missing_file_raises PASSED     [ 12%]
tests/test_io_mocks.py::test_load_dataset_with_monkeypatch PASSED        [ 18%]
tests/test_io_mocks.py::test_save_dataset_calls_to_csv_without_index PASSED [ 25%]
tests/test_io_mocks.py::test_preprocess_pipeline_uses_loaded_data PASSED [ 31%]
tests/test_preprocessing.py::test_normalize_scales_to_unit_range PASSED  [ 37%]
tests/test_preprocessing.py::test_normalize_does_not_mutate_input PASSED [ 43%]
tests/test_prepr

**Useful flags:** `-v` (verbose) / `-q` (quiet dots) · `-k "normalize"` (match by name) ·
`-x` (stop at first failure) · `--lf` (re-run only last-failed).

In [30]:
# Run only the normalize tests
!{sys.executable} -m pytest -q -k "normalize" 

...                                                                      [100%]
3 passed, 13 deselected in 0.02s


### Red → green
Watch a failing test go red, then fix the expectation and watch it go green — the core loop, and
exactly what a PR check enforces. (The sandbox seeds this as `feature/clip-outliers`: a
`clip_outliers()` with an unfinished body and a failing spec — implement it with `Series.clip()` to go
green, then open a PR and watch CI run it.)

In [31]:
# A deliberately wrong expectation -> RED
Path("tests/test_regression.py").write_text(
    "import pandas as pd\n"
    "from preprocessing import normalize\n\n"
    "def test_normalized_min_is_zero():\n"
    "    df = pd.DataFrame({'a': [10.0, 20.0, 30.0]})\n"
    "    assert normalize(df, ['a'])['a'].iloc[0] == 0.5   # wrong on purpose\n"
)
!{sys.executable} -m pytest -q tests/test_regression.py; echo "exit code: $?" 

F                                                                        [100%]
=================================== FAILURES ===================================
_________________________ test_normalized_min_is_zero __________________________

    def test_normalized_min_is_zero():
        df = pd.DataFrame({'a': [10.0, 20.0, 30.0]})
>       assert normalize(df, ['a'])['a'].iloc[0] == 0.5   # wrong on purpose
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       assert np.float64(0.0) == 0.5

tests/test_regression.py:6: AssertionError
=========================== short test summary info ============================
FAILED tests/test_regression.py::test_normalized_min_is_zero - assert np.float64(0.0) == 0.5
1 failed in 0.03s
exit code: 1


In [32]:
# Correct the expectation -> GREEN
Path("tests/test_regression.py").write_text(
    "import pandas as pd\n"
    "from preprocessing import normalize\n\n"
    "def test_normalized_min_is_zero():\n"
    "    df = pd.DataFrame({'a': [10.0, 20.0, 30.0]})\n"
    "    assert normalize(df, ['a'])['a'].iloc[0] == 0.0\n"
)
!{sys.executable} -m pytest -q tests/test_regression.py; echo "exit code: $?" 

.                                                                        [100%]
1 passed in 0.00s
exit code: 0


## Coverage — measure what the tests exercise

```bash
pytest --cov=. --cov-report=term-missing
```

Coverage reports which lines ran during the tests and which did not — great for finding gaps.

> **A compass, not a scoreboard.** A line can *run* without being *checked*: 100% coverage ≠ correct.
> Configure it once in `pyproject.toml` (omit `tests/*` and CLI entrypoints) and test *behaviour*.

In [33]:
# The same coverage line CI runs, against our scratch package
!{sys.executable} -m pytest -q --cov=preprocessing --cov-report=term-missing

.................                                                        [100%]
================================ tests coverage ================================
______________ coverage: platform darwin, python 3.12.13-final-0 _______________

Name               Stmts   Miss  Cover   Missing
------------------------------------------------
preprocessing.py      41      8    80%   26-30, 35-37
------------------------------------------------
TOTAL                 41      8    80%
17 passed in 0.07s


## Testing ML pipelines — why it matters and what to test

Everything so far tested ordinary Python. Machine-learning pipelines need the *same* discipline, but
they fail in a way ordinary code usually does not: **silently**. A bug rarely throws - it just makes
the model a little worse, ships a plausible-but-wrong answer, or quietly skips a step. Nothing turns
red; the metrics dashboard looks fine; a subtly degraded model reaches production. Unit tests are how
you turn those silent failures into loud ones.

Why ML pipelines need testing *more*, not less:

- **Data is a second input.** Your code can be perfect and still produce garbage because a column was
  renamed, a scale changed, or NaNs crept in. Tests pin the assumptions the model was built on.
- **Training/serving skew.** The classic ML bug: a transform (scaling, tokenisation, feature order)
  is applied at training time but forgotten - or applied differently - at inference. A save→load→predict
  round-trip test catches it before users do. (The sandbox's `test_model.py` does exactly this.)
- **Refactors and dependency bumps move numbers.** A pandas or torch upgrade can subtly change a
  computation. A test that pins expected outputs tells you *immediately*, not three experiments later.
- **Aggregate metrics hide specific regressions.** A model can improve on average while breaking the
  ten inputs you most care about. Overall accuracy won't catch it - a **behavioral test** on those
  exact inputs will.
- **Non-determinism is easy to introduce.** A missing seed, an unstable sort, a dict-ordering
  assumption. A "same seed → same output" test makes reproducibility a checked property, not a hope.

> **Unit tests are not model evaluation.** Evaluation measures *quality* (accuracy, F1, BLEU) on a
> large held-out set and is inherently statistical. **Unit tests** check *deterministic logic and
> contracts* - shapes, transforms, scaling, output schemas, and specific known-good behaviors - and
> run in seconds on every push. You want both; this section is about the fast, deterministic layer
> that gates every PR in CI.


### What to test in an ML pipeline

A practical map, from the most unit-testable (top) to the most eval-like (bottom):

| Layer | What you assert | In the sandbox |
| ----- | --------------- | -------------- |
| **Data & preprocessing** | schema/columns present, no NaN leakage, ranges after scaling, deterministic split, no train/test overlap | `test_preprocessing.py` (`drop_missing`, `normalize`, `train_test_split`) |
| **Feature invariants** | `normalize` → [0, 1]; `standardize` → mean≈0/std≈1; constant column → zeros; **input not mutated** | `test_normalize_does_not_mutate_input`, parametrized cases |
| **Model architecture & math** | forward output shape == batch size; layer shapes match config; scaling/un-scaling applied correctly | `test_model.py` (`RegressionMLP`, `Artifacts.predict`) |
| **Save/load round-trip** | reload gives *identical* predictions — guards against training/serving skew | `test_roundtrip_predictions_are_identical` |
| **Determinism** | same seed → identical weights/outputs | `test_same_seed_gives_identical_weights` |
| **Behavioral / known-answer** | specific critical inputs get the response you *know* is correct | *(the focus below)* |
| **Guardrails / safety** | disallowed inputs are refused or flagged; PII is always redacted; output is always valid | *(the focus below)* |
| **Threshold / regression** | accuracy on a small fixed eval set stays above a floor (a smoke test, run in CI; full eval runs elsewhere) | — |


### Behavioral tests: pin the model's known-good answers

This is the part that matters most for a system built around an AI model. You have a set of inputs
whose correct output you already **know** - and must not regress as the model, prompt, or data
changes. A behavioral test locks each one in. Four shapes, borrowed from the *CheckList* methodology
([Ribeiro et al., 2020](https://aclanthology.org/2020.acl-main.442/)):

- **Minimum functionality (known-answer):** for input X, the output must be Y. Your golden set - the
  handful of cases the product absolutely must get right.
- **Invariance:** a change that *shouldn't* alter the output doesn't - paraphrase, change of case,
  an appended neutral clause, reordered fields. If it flips the label, that's a bug.
- **Directional expectation:** a change that *should* move the output in a known direction does -
  adding strongly positive words must not *lower* a positivity score.
- **Guardrail / safety:** for critical or disallowed inputs, the system does the safe thing every
  time - refuses, flags, or redacts. These never regress silently.

The runnable cells below build a tiny, deterministic "AI service" (rule-based, no API key, no
network — so it runs anywhere and in CI) and write each of these test shapes against it. The point is
the **test pattern**, which is identical whether the thing under test is a rule, a scikit-learn model,
a PyTorch network, or a call to a large language model.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

MLLAB = Path.home() / "Desktop" / "ml_testing_demo"
if MLLAB.exists():
    shutil.rmtree(MLLAB)
(MLLAB / "tests").mkdir(parents=True)
os.chdir(MLLAB)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=False)

# pytest config so `import ai_service` resolves from the project root (the gotcha from earlier)
Path("pyproject.toml").write_text(
    '[tool.pytest.ini_options]\npythonpath = ["."]\ntestpaths = ["tests"]\n'
)
print("ML test lab:", MLLAB)

#### The system under test
A stand-in for an AI model: a sentiment scorer and a support-ticket triager, plus a PII-redaction
guardrail. Rule-based and deterministic on purpose — in a real project this same surface would wrap a
trained model or an LLM call, and the tests would not change.

In [ ]:
%%writefile ai_service.py
"""A tiny, deterministic stand-in for an AI service (no model weights, no network).

Swap the internals for a real model or an LLM call and the tests around it stay the same.
"""

import re

_POSITIVE = {"love", "great", "excellent", "amazing", "good", "wonderful", "happy", "fast"}
_NEGATIVE = {"hate", "terrible", "awful", "bad", "worst", "broken", "angry", "slow"}

_INTENTS = {"billing_dispute", "account_recovery", "technical", "general"}
_PRIORITIES = {"low", "medium", "high"}

_EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")


def _tokens(text):
    return re.findall(r"[a-z']+", text.lower())


def score_sentiment(text: str) -> float:
    """Signed sentiment score: +1 per positive word, -1 per negative word."""
    toks = _tokens(text)
    return sum(t in _POSITIVE for t in toks) - sum(t in _NEGATIVE for t in toks)


def classify_sentiment(text: str) -> str:
    """Map the score to a label. Case-insensitive; neutral when the score is 0."""
    s = score_sentiment(text)
    return "positive" if s > 0 else "negative" if s < 0 else "neutral"


def triage(text: str) -> dict:
    """Route a support ticket. Returns {'intent': ..., 'priority': ...}."""
    t = text.lower()
    if "refund" in t or "charged twice" in t or "double charged" in t:
        intent, priority = "billing_dispute", "high"
    elif "password" in t or "can't log in" in t or "locked out" in t:
        intent, priority = "account_recovery", "medium"
    elif "error" in t or "crash" in t or "bug" in t:
        intent, priority = "technical", "medium"
    else:
        intent, priority = "general", "low"
    return {"intent": intent, "priority": priority}


def redact_pii(text: str) -> str:
    """Guardrail: replace any email address with a placeholder."""
    return _EMAIL.sub("[EMAIL]", text)

#### The behavioral test suite
Each class of test from above, as ordinary pytest — fixtures, parametrization, `raises`, and
`approx`, exactly the tools from earlier in this pillar.

In [ ]:
%%writefile tests/test_behavior.py
"""Behavioral tests for the AI service: known-answer, invariance, directional, guardrail."""

import pytest

from ai_service import classify_sentiment, score_sentiment, triage, redact_pii


# --- 1. Minimum functionality / known-answer -------------------------------
# The golden set: inputs whose correct label we KNOW and must never regress.
@pytest.mark.parametrize(
    "text, expected",
    [
        ("I love this, it is excellent", "positive"),
        ("this is terrible and broken", "negative"),
        ("the meeting is at noon", "neutral"),
    ],
)
def test_sentiment_known_answers(text, expected):
    assert classify_sentiment(text) == expected


def test_critical_response_is_pinned():
    """A response we KNOW must be right: a double-charge is a high-priority billing dispute."""
    assert triage("I was charged twice and I want a refund") == {
        "intent": "billing_dispute",
        "priority": "high",
    }


# --- 2. Invariance: outputs that must NOT change ---------------------------
@pytest.mark.parametrize("neutral_tail", ["", " The meeting is on Tuesday.", " Please advise."])
def test_sentiment_invariant_to_neutral_clause(neutral_tail):
    base = "I love this product"
    assert classify_sentiment(base + neutral_tail) == "positive"


def test_sentiment_invariant_to_case():
    assert classify_sentiment("I LOVE THIS") == classify_sentiment("i love this")


# --- 3. Directional expectation: outputs that must move a known way --------
def test_adding_positive_words_never_lowers_score():
    base = "the app is good"
    stronger = base + " and absolutely wonderful and amazing"
    assert score_sentiment(stronger) >= score_sentiment(base)


# --- 4. Guardrail / safety: must hold for EVERY input ----------------------
@pytest.mark.parametrize(
    "text",
    [
        "email me at jane.doe@example.com",
        "contact: a.b+tag@sub.domain.co.uk please",
        "no email here",
    ],
)
def test_pii_is_always_redacted(text):
    out = redact_pii(text)
    assert "@" not in out                      # no raw email survives
    if "@" in text:
        assert "[EMAIL]" in out                # and it was actually masked


# --- 5. Output contract / schema: the shape is always valid ----------------
@pytest.mark.parametrize(
    "text",
    ["I want a refund", "I forgot my password", "the app keeps crashing", "hello there"],
)
def test_triage_output_schema(text):
    out = triage(text)
    assert set(out) == {"intent", "priority"}
    assert out["intent"] in {"billing_dispute", "account_recovery", "technical", "general"}
    assert out["priority"] in {"low", "medium", "high"}

In [ ]:
# Run the behavioral suite — the same green/red signal CI uses on every PR
!{sys.executable} -m pytest -v

# Pillar 3 — CI/CD with GitHub Actions

## What is CI/CD — and why hand it to a robot

- **CI (Continuous Integration)** — on every push and pull request, automatically install, lint, and
  **run the whole test suite** on a clean machine you do not control. Nobody can forget to test.
- **CD (Continuous Delivery)** — once green, automatically **build and publish** the artifact (here, a
  Docker image to the GitHub Container Registry).

> **"Works on my machine" dies here.** CI runs your tests on a clean machine, on every Python version
> you support, for every contributor — and becomes the **required check** that guards `main` /
> `staging`.

**Vocabulary:** a **workflow** contains **jobs**, which contain **steps**. Jobs run in parallel by
default; steps run in order. Workflows are YAML files under `.github/workflows/`. These run on
GitHub's servers, so we *read* them here rather than execute them - but the `pytest` they run is the
exact one you ran in Pillar 2.


## Anatomy of a workflow — the sandbox's `ci.yml`

```yaml
name: CI

on:                       # WHEN it runs
  push:
    branches: [main, staging]
  pull_request:           # every PR, automatically
  workflow_dispatch:      # a manual "Run" button in the Actions tab

concurrency:              # cancel superseded runs on the same ref -> save CI minutes
  group: ci-${{ github.ref }}
  cancel-in-progress: true

jobs:
  test:                                        # WHAT it does
    name: Test (Python ${{ matrix.python-version }})
    runs-on: ubuntu-latest                     # a fresh Ubuntu VM per job
    strategy:
      fail-fast: false                         # if 3.11 fails, 3.12 still runs
      matrix:
        python-version: ["3.11", "3.12"]       # run the whole job once PER version, in parallel
    steps:
      - uses: actions/checkout@v4              # clone the repo
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip                           # cache wheels between runs
      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install torch --index-url https://download.pytorch.org/whl/cpu   # CPU-only wheel
          pip install -r requirements.txt
      - name: Run test suite with coverage
        run: pytest -v --cov=. --cov-report=term-missing --cov-report=xml
      - name: Upload coverage artifact
        if: always()                           # upload even if tests failed
        uses: actions/upload-artifact@v4
        with:
          name: coverage-${{ matrix.python-version }}
          path: coverage.xml
```

| Piece | Why it matters |
| ----- | -------------- |
| `on:` push + pull_request | Feature branches don't burn minutes on every push, but **opening a PR runs CI**. |
| `strategy.matrix` | Tests across **multiple Python versions** in parallel — catches version-specific breakage. |
| `fail-fast: false` | You see *all* failures, not just the first version to break. |
| `cache: pip` | Reuses downloaded wheels → much faster runs. |
| CPU-only torch | Avoids a multi-GB CUDA download in CI (and 3.14 is skipped — no torch wheels yet). |
| final `pytest` step | **The job's pass/fail is this step's exit code** — a failing test = a red check that blocks the PR. |
| `concurrency` | Kills stale runs when you push again, saving minutes. |
| `uses:` vs `run:` | `uses:` pulls a prebuilt action (pinned with `@v4`); `run:` executes a shell command. |


## Closing the loop — required checks make it real

```
open PR  ──►  CI runs tests  ──►  green = merge allowed  ──►  merge to main  ──►  CD builds & publishes image
                    │
                    └─►  red = merge blocked
```

All three pillars snap together: a **PR** (Pillar 1) triggers **tests** (Pillar 2) in **CI**
(Pillar 3); a red run blocks the merge, a green run lets it through; merging to `main` ships the
image; a `v*` tag marks a reproducible release.

**Enable it:** Settings → Branches → protect `main`/`staging` → require the `test` check to pass, and
forbid direct/force pushes.

> **See it live.** Open the sandbox's **Actions** tab: CI runs on `main`/`staging` and every PR;
> `feature/clip-outliers` shows a red run *by design* until the exercise is finished.


## Live demo — a real pull request, real CI (red → green)

Everything above explained the workflow files. This section **runs it for real**: it opens an actual
pull request on the sandbox, lets GitHub Actions run the unit-test suite, and watches the checks go
**red** (the unfinished `clip_outliers`) and then **green** (after the fix) — with the failing and
passing pytest output printed right here in the notebook.

**What it drives:** the planted `feature/clip-outliers` branch → a PR into `staging`. That branch
ships `clip_outliers()` with an intentionally failing spec, so it is the perfect subject for showing
that *unit tests gate a pull request*.

**Prerequisites (already satisfied on the presenter machine):**
- The GitHub CLI, authenticated with `repo` + `workflow` scopes: check with `gh auth status`.
- Write access to the sandbox (to push the fix and open the PR).

> **Before you run this — wake up GitHub Actions (one time).** A workflow's `pull_request` trigger
> stays **dormant until the workflow has run at least once**. On a fresh sandbox, opening a PR may
> therefore start **no** CI run at all (the PR shows zero checks). Wake it up once: push any commit to
> `staging`/`main`, or open the repo's **Actions** tab → **CI** → **Run workflow**. After that first
> run, PRs trigger CI inline as described here.
>
> **If you skip the wake-up, the cells below still work.** When no PR-triggered run appears within
> ~30 seconds, the run-finder automatically falls back to a manual `workflow_dispatch` on the branch —
> same workflow, same matrix, same **red → green** result. The only difference is that a dispatched
> run executes the branch's current tip and does **not** appear as an inline check on the PR.

> **A note on "blocking" the merge.** In this sandbox `staging` is **not** branch-protected, so a red
> run shows a red ✗ but does not literally prevent merging. To make a red run *block* the merge,
> add a required status check (Settings → Branches → rule for `staging` → require
> `Test (Python 3.11)` / `Test (Python 3.12)`), or run the optional protection cell near the end.
>
> **Scope.** We watch the fast **`ci.yml`** run (pip + pytest across 3.11/3.12). The
> `docker-publish.yml` workflow also runs on the PR (build + test-in-image, no publish); it appears
> as an extra check and is fine to ignore for this segment.

In [ ]:
import os, sys, re, json, time, shutil, subprocess
from pathlib import Path

REPO = "amirrezayasami-amii/amii-workshop-sandbox"
BASE = "staging"
HEAD = "feature/clip-outliers"
DEMO = Path.home() / "Desktop" / "cicd_live_demo"


def run(args, echo=True):
    """Run a command, optionally print its output, and return the CompletedProcess."""
    r = subprocess.run(args, text=True, capture_output=True)
    if echo:
        if r.stdout:
            print(r.stdout, end="")
        if r.stderr:
            print(r.stderr, end="")
    return r


def gh_json(args):
    """Run a `gh ... --json ...` command and parse the result.

    Defensive: returns [] on empty or non-JSON output instead of raising, so a transient
    gh hiccup never crashes the demo — the callers below simply treat it as "nothing yet".
    """
    out = run(["gh", *args], echo=False).stdout.strip()
    if not out:
        return []
    try:
        return json.loads(out)
    except json.JSONDecodeError:
        return []


def dispatch_ci():
    """Fallback trigger: run ci.yml manually on HEAD via workflow_dispatch.

    A workflow's `pull_request` trigger stays dormant until the workflow has run at least once
    (see the wake-up note above), so on a fresh sandbox the PR may start no run. A dispatched run
    executes the branch's current tip, so it is still red before the fix and green after it — it
    just doesn't show as an inline check on the PR.
    """
    run(["gh", "workflow", "run", "ci.yml", "--repo", REPO, "--ref", HEAD], echo=False)


def ci_run_for(head_sha, timeout=180, dispatch_after=30):
    """Return the ci.yml run for the given head SHA.

    Prefer a run started by the PR (push / pull_request). If none appears within
    `dispatch_after` seconds, fall back to a manual workflow_dispatch and keep polling.
    """
    dispatched = False
    for elapsed in range(0, timeout, 5):
        for r in gh_json(["run", "list", "--repo", REPO, "--workflow", "ci.yml",
                          "--branch", HEAD, "-L", "8",
                          "--json", "databaseId,headSha,status,conclusion,url,event"]):
            if r["headSha"].startswith(head_sha):
                return r
        if not dispatched and elapsed >= dispatch_after:
            print("No PR-triggered run yet — dispatching ci.yml manually (see the wake-up note).")
            dispatch_ci()
            dispatched = True
        time.sleep(5)
    return None


# Confirm gh is authenticated (needs 'repo' + 'workflow' scopes)
run(["gh", "auth", "status"])

# Fresh clone of the sandbox — leaves ~/amii-workshop-sandbox and any practice copy untouched.
# Step out to a stable dir first: on a re-run the kernel may be cwd'd *inside* DEMO, and
# deleting the directory you're standing in breaks git ("Unable to read current working directory").
os.chdir(Path.home())
if DEMO.exists():
    shutil.rmtree(DEMO)
run(["gh", "repo", "clone", REPO, str(DEMO), "--", "-q"])
os.chdir(DEMO)
run(["git", "switch", HEAD])
print("\nReady in", DEMO, "on", HEAD)

In [ ]:
# Open a PR from feature/clip-outliers into staging — reuse an existing open one if present.
# `gh pr list` can lag right after a create (eventual consistency), so look the PR up with a
# short retry via `gh pr view`, and treat "already exists" on create as success.
def _find_pr():
    for _ in range(8):
        v = gh_json(["pr", "view", HEAD, "--repo", REPO, "--json", "number,url,state"])
        if isinstance(v, dict) and v.get("state") == "OPEN":
            return v
        time.sleep(2)
    return None


pr = _find_pr()
if pr is None:
    run(["gh", "pr", "create", "--repo", REPO, "--base", BASE, "--head", HEAD,
         "--title", "feat: implement clip_outliers (workshop CI demo)",
         "--body", "Ships clip_outliers() with a failing unit test, then the fix. "
                   "Watch CI go red, then green."])
    pr = _find_pr()
assert pr is not None, "Could not locate the PR after creating it."
PR, PR_URL = pr["number"], pr["url"]
print("PR:", PR_URL)

### Step 1 — CI runs the unit tests and goes RED

Opening the PR fires the `pull_request` trigger, so `ci.yml` starts automatically. The suite includes
`tests/test_clip_outliers.py`, which fails because `clip_outliers()` is still unfinished — so the run
goes red. The cell below **watches the run to completion** and then prints the failing pytest lines
straight from the CI log.

In [ ]:
# Watch the CI run for the current head — expected to FAIL (clip_outliers is unfinished).
head_sha = run(["git", "rev-parse", "HEAD"], echo=False).stdout.strip()
r = ci_run_for(head_sha[:10])
print("CI run:", r["url"] if r else "(not found yet — re-run this cell)")
if r:
    # --exit-status makes gh return non-zero when the run fails; that red result is expected here.
    run(["gh", "run", "watch", str(r["databaseId"]), "--repo", REPO,
         "--exit-status", "--interval", "15"])
    log = run(["gh", "run", "view", str(r["databaseId"]), "--repo", REPO, "--log-failed"],
              echo=False).stdout
    print("\n--- CI unit-test results (the failing step) ---")
    for line in log.splitlines():
        msg = re.sub(r"^.*?\S+Z ", "", line)          # strip the job / step / timestamp prefix
        if any(k in msg for k in ("clip_outliers", "assert", "Error", "FAILED",
                                  " passed", " failed")):
            print(msg[:200])

### Step 2 — push the fix; CI goes GREEN

Now we implement `clip_outliers()` in the clone, commit, and push. Pushing to the PR branch fires the
`pull_request: synchronize` event, so CI runs again on the new commit — and this time the whole suite
passes.

In [ ]:
# Implement clip_outliers in the clone, commit, and push -> triggers a fresh CI run on the PR
pp = DEMO / "preprocessing.py"
src = pp.read_text()
fixed = src.replace(
    "    out = df.copy()\n"
    "    # TODO: clip each column to its lower/upper quantiles with Series.clip().\n"
    "    return out",
    "    out = df.copy()\n"
    "    for col in columns:\n"
    "        lo, hi = out[col].quantile(lower), out[col].quantile(upper)\n"
    "        out[col] = out[col].clip(lower=lo, upper=hi)\n"
    "    return out",
)
assert fixed != src, "TODO block not found — is the clone on feature/clip-outliers?"
pp.write_text(fixed)
run(["git", "add", "preprocessing.py"])
run(["git", "commit", "-m", "feat: implement clip_outliers to cap extremes at quantiles"])
run(["git", "push"])
print("Pushed the fix; CI will re-run on the PR.")

In [ ]:
# Watch the new CI run -> expected to PASS; show the passing unit-test results + PR checks.
head_sha = run(["git", "rev-parse", "HEAD"], echo=False).stdout.strip()
r = ci_run_for(head_sha[:10])
print("CI run:", r["url"] if r else "(not found yet — re-run this cell)")
if r:
    run(["gh", "run", "watch", str(r["databaseId"]), "--repo", REPO,
         "--exit-status", "--interval", "15"])
    log = run(["gh", "run", "view", str(r["databaseId"]), "--repo", REPO, "--log"],
              echo=False).stdout
    print("\n--- CI unit-test results (all green) ---")
    for line in log.splitlines():
        msg = re.sub(r"^.*?\S+Z ", "", line)
        if "clip_outliers.py::" in msg or re.search(r"\d+ passed", msg):
            print(msg[:200])
print("\n--- PR check summary ---")
run(["gh", "pr", "checks", str(PR), "--repo", REPO])
print("\nGreen. The same unit tests you ran in Pillar 2 now pass in CI.  PR:", PR_URL)

### Optional — make a red run actually block the merge

Run this only if you want the demo to show the merge being *blocked* while CI is red. It adds a
required status check on `staging` (admin only). Re-running the whole demo afterwards will show the
green PR become mergeable.

In [ ]:
# OPTIONAL (admin) — require the CI checks on staging so a red run blocks the merge.
payload = {
    "required_status_checks": {"strict": True,
                               "contexts": ["Test (Python 3.11)", "Test (Python 3.12)"]},
    "enforce_admins": False,
    "required_pull_request_reviews": None,
    "restrictions": None,
}
res = subprocess.run(["gh", "api", "--method", "PUT",
                      f"repos/{REPO}/branches/{BASE}/protection", "--input", "-"],
                     input=json.dumps(payload), text=True, capture_output=True)
print("protection set" if res.returncode == 0 else res.stderr[:300])
# To relax again later:
# subprocess.run(["gh", "api", "--method", "DELETE",
#                 f"repos/{REPO}/branches/{BASE}/protection"])

### Reset the demo (repeatable)

Restores the sandbox to its seeded state so the demo can be run again: force-pushes
`feature/clip-outliers` back to its original failing commit and closes the PR.

In [ ]:
# Restore feature/clip-outliers to its seeded (failing) commit and close any open PR.
# Self-sufficient: looks the PR up itself (no dependency on the `PR` variable) and resets the
# branch ref via the API (no local clone needed), so it works even run on its own.
ORIGINAL_SHA = "1bfb4c6ca33c024ec00725d48ec89614155e81b6"   # seeded tip of feature/clip-outliers
_pr = gh_json(["pr", "view", HEAD, "--repo", REPO, "--json", "number,state"])
if isinstance(_pr, dict) and _pr.get("state") == "OPEN":
    run(["gh", "pr", "close", str(_pr["number"]), "--repo", REPO])
run(["gh", "api", "-X", "PATCH", f"repos/{REPO}/git/refs/heads/{HEAD}",
     "-f", f"sha={ORIGINAL_SHA}", "-F", "force=true"], echo=False)
print(f"Reset {HEAD} to {ORIGINAL_SHA[:8]}; any open demo PR closed.")

# Putting it together — the live lab

Everything above converges on one hands-on loop. In the sandbox ([https://github.com/amirrezayasami-amii/amii-workshop-sandbox](https://github.com/amirrezayasami-amii/amii-workshop-sandbox)):

1. `git clone https://github.com/amirrezayasami-amii/amii-workshop-sandbox && cd amii-workshop-sandbox` — you land on `staging`.
2. **(Git)** `git switch -c feature/<your-name>`; add your line to `PARTICIPANTS.md`; commit; push;
   open a PR into `staging`.
3. **(Git)** Merge `feature/tune-lr` into `staging` and **resolve the conflict** in `config.py` by
   hand.
4. **(Test)** Switch to `feature/clip-outliers`; make `tests/test_clip_outliers.py` go from red to
   green.
5. **(CI/CD)** Push it, open the PR, and watch **GitHub Actions** run your tests on the run page.
6. **(Bonus)** Rebase `feature/faster-epochs` onto `staging`, then `git push --force-with-lease`.

# Wrap-up

**Git &amp; GitHub** - read the graph first; `main` stable, branch off `staging`; rebase your own work
and let `reflog` save you; small PRs, kind reviews.
**Testing** - fixtures, parametrize, mock; test behaviour, not lines; `approx`/`raises`; red → green →
refactor.
**CI/CD** - tests on every PR; matrix + cache + coverage; least-privilege secrets; a green gate before
publish.

> **The one-sentence takeaway.** A **pull request** that must pass **automated tests** in **CI**
> before it can merge is the entire game — everything here serves that loop.

**Resources**
- The sandbox (clone, break, practise): [https://github.com/amirrezayasami-amii/amii-workshop-sandbox](https://github.com/amirrezayasami-amii/amii-workshop-sandbox)
- Workshop `docs/` — copy-pasteable modules on Git, pytest, and CI/CD.
- [Pro Git (free book)](https://git-scm.com/book) · [docs.pytest.org](https://docs.pytest.org) ·
  [docs.github.com/actions](https://docs.github.com/actions)
- [A successful Git branching model](https://nvie.com/posts/a-successful-git-branching-model/) ·
  [Conventional Commits](https://www.conventionalcommits.org) · [Oh Sh*t, Git!?!](https://ohshitgit.com)